# Laya + LoRA on the top encoder layers: QQP duplicate questions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalmysore/layaMOE/blob/main/notebooks/laya_lora_qqp.ipynb)

Training only Laya's decision head did not move QQP (see the README): the head alone can't learn what the frozen encoder doesn't already represent. This notebook lets the **top encoder layers adapt with LoRA** while the rest of the encoder stays frozen, the step that took GLiNER2.5-Decide from 69.7% to 80.3% on the same kind of slice.

| | |
|---|---|
| Base | [`convaiinnovations/laya-typed-decisions`](https://huggingface.co/convaiinnovations/laya-typed-decisions) (ModernBERT-large, 28 layers) |
| Trainable | LoRA (rank 16) on `Wqkv`, `attn.Wo`, `mlp.Wi`, `mlp.Wo` of the **top 4 layers** + a copy of the decision head |
| Frozen | embeddings and the bottom 24 layers, which stay shareable across MoE experts |
| Train | 1,500 balanced pairs from QQP **train** (questions that appear in the eval slice are excluded) |
| Eval | 1,500 balanced pairs from QQP **validation**, `random_state=42` (the same slice as `scripts/eval_qqp.py`) |

**Runtime → Change runtime type → T4 GPU.** About 10 minutes on a free T4. On CPU it works, but slowly (set `SMOKE = True` for a quick check).

## 1. Setup

In [1]:
%%capture
!pip install -q laya==0.3.4 safetensors huggingface_hub scipy pandas pyarrow

In [2]:
import os, sys, subprocess
REPO = "layaMOE"
if os.path.exists("laya_moe") and os.path.exists("data/eval"):
    REPO = "."
elif os.path.exists(os.path.join("..", "laya_moe")):
    REPO = ".."
elif not os.path.exists(os.path.join(REPO, "laya_moe")):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/vishalmysore/layaMOE.git", REPO], check=True)
sys.path.insert(0, os.path.abspath(REPO))

import json, math, random, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from laya_moe.core import ExpertHead, load_base, make_item, label_index
sys.path.insert(0, os.path.join(os.path.abspath(REPO), "scripts"))
from eval_qqp import load_slice, metrics, QUESTION

SMOKE = os.environ.get("SMOKE") == "1"   # tiny run to check the notebook end to end
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
cfg = dict(
    top_layers=1 if SMOKE else 4,   # LoRA on the top N of 28 encoder layers
    rank=16, alpha=32, dropout=0.05,
    lr_lora=2e-4, lr_head=2e-4, epochs=1 if SMOKE else 3, bs=16, seed=0,
    train_per_class=16 if SMOKE else 750, eval_per_class=20 if SMOKE else 750,
)
random.seed(cfg["seed"]); torch.manual_seed(cfg["seed"])
print("device", DEVICE, "| torch", torch.__version__, "| smoke" if SMOKE else "", cfg)

device cpu | torch 2.14.0+cu130  {'top_layers': 4, 'rank': 16, 'alpha': 32, 'dropout': 0.05, 'lr_lora': 0.0002, 'lr_head': 0.0002, 'epochs': 3, 'bs': 16, 'seed': 0, 'train_per_class': 750, 'eval_per_class': 750}


## 2. Data
Training pairs come from QQP's train split; any pair sharing a question with the eval slice is dropped (`scripts/gen_qqp_data.py`).

In [3]:
train_path = os.path.join(REPO, "data", "train", "pairs", "qqp.jsonl")
subprocess.run([sys.executable, os.path.join(REPO, "scripts", "gen_qqp_data.py"), "--per-class", str(cfg["train_per_class"])], check=True)
rows = [json.loads(l) for l in open(train_path, encoding="utf-8")]
random.shuffle(rows)
n_val = max(4, len(rows) // 10)
train_rows, val_rows = rows[n_val:], rows[:n_val]

ev = load_slice(n_per_class=cfg["eval_per_class"])
eval_states = [{"question1": a, "question2": b} for a, b in zip(ev.question1, ev.question2)]
eval_y = ev.label.values.astype(bool)
print(f"train {len(train_rows)} | val {len(val_rows)} | eval {len(ev)} ({int(eval_y.sum())} duplicate)")

wrote 1500 pairs (750 duplicate) to /home/claude/layaMOE/notebooks/../scripts/../data/train/pairs/qqp.jsonl; 8683 train pairs dropped for sharing a question with the eval slice


train 1350 | val 150 | eval 1500 (750 duplicate)


## 3. Model: frozen encoder, LoRA on the top layers, trainable decision head

In [4]:
base = load_base()
tok, bcfg = base.tok, base.cfg
encoder = base.model.encoder

class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))); W frozen, A and B trained. B starts at zero, so training starts from Laya exactly."""
    def __init__(self, base_linear, r, alpha, dropout):
        super().__init__()
        self.base = base_linear
        self.A = nn.Linear(base_linear.in_features, r, bias=False)
        self.B = nn.Linear(r, base_linear.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + self.B(self.A(self.drop(x))) * self.scale

for p in encoder.parameters():
    p.requires_grad_(False)
n_layers = len(encoder.layers)
lora_layers = list(range(n_layers - cfg["top_layers"], n_layers))
for li in lora_layers:
    L = encoder.layers[li]
    L.attn.Wqkv = LoRALinear(L.attn.Wqkv, cfg["rank"], cfg["alpha"], cfg["dropout"])
    L.attn.Wo = LoRALinear(L.attn.Wo, cfg["rank"], cfg["alpha"], cfg["dropout"])
    L.mlp.Wi = LoRALinear(L.mlp.Wi, cfg["rank"], cfg["alpha"], cfg["dropout"])
    L.mlp.Wo = LoRALinear(L.mlp.Wo, cfg["rank"], cfg["alpha"], cfg["dropout"])

head = ExpertHead(base.model)          # copy of Laya's decision head (type embedding, 2 layers, scorer)
encoder.to(DEVICE); head.to(DEVICE)
lora_params = [p for n, p in encoder.named_parameters() if p.requires_grad]
print(f"LoRA on layers {lora_layers[0]}-{lora_layers[-1]}: {sum(p.numel() for p in lora_params)/1e6:.2f}M params | "
      f"head {sum(p.numel() for p in head.parameters())/1e6:.1f}M | frozen encoder {sum(p.numel() for p in encoder.parameters() if not p.requires_grad)/1e6:.0f}M")

LoRA on layers 24-27: 1.03M params | head 26.5M | frozen encoder 395M


In [5]:
def to_item(state, qdef, label=None):
    it = make_item(tok, bcfg, state, qdef)
    it["label"] = label
    return it

def collate(items):
    L = max(len(i["ids"]) for i in items); K = max(len(i["markers"]) for i in items); n = len(items)
    ids = torch.full((n, L), tok.pad_token_id, dtype=torch.long); att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, K), dtype=torch.long); mmask = torch.zeros((n, K), dtype=torch.bool)
    for i, it in enumerate(items):
        ids[i, :len(it["ids"])] = torch.tensor(it["ids"]); att[i, :len(it["ids"])] = 1
        mpos[i, :len(it["markers"])] = torch.tensor(it["markers"]); mmask[i, :len(it["markers"])] = True
    qt = torch.tensor([it["qtype"] for it in items])
    return [t.to(DEVICE) for t in (ids, att, mpos, mmask, qt)]

def forward(items):
    ids, att, mpos, mmask, qt = collate(items)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
        h = encoder(input_ids=ids, attention_mask=att).last_hidden_state
        logits = head(h, att, mpos, mmask, qt)
    return logits.float(), mmask

train_items = [to_item(r["state"], r["questions"]["duplicate"], label_index(r["questions"]["duplicate"], r["expected"]["duplicate"])) for r in train_rows]
val_items = [to_item(r["state"], r["questions"]["duplicate"], label_index(r["questions"]["duplicate"], r["expected"]["duplicate"])) for r in val_rows]
eval_items = [to_item(s, QUESTION) for s in eval_states]

@torch.no_grad()
def p_yes(items, bs=64):
    encoder.eval(); head.eval(); out = []
    for i in range(0, len(items), bs):
        logits, _ = forward(items[i:i + bs])
        out.append(torch.softmax(logits[:, :2], -1)[:, 1].cpu())
    return torch.cat(out).numpy()

def acc_on(items):
    y = np.array([it["label"] == 1 for it in items])
    return float(((p_yes(items) >= 0.5) == y).mean())

## 4. Zero-shot baseline
With LoRA's `B` at zero and the head copied from Laya, this is exactly the original model. It should match `results/qqp_eval.json` (77.2% on the full slice).

In [6]:
t0 = time.time()
p0 = p_yes(eval_items)
zero = metrics(p0, eval_y)
print(f"zero-shot: accuracy {zero['accuracy']:.1%}  F1 {zero['f1']:.3f}  ({time.time() - t0:.0f}s)")

zero-shot: accuracy 77.2%  F1 0.767  (818s)


## 5. Train LoRA + head

In [7]:
opt = torch.optim.AdamW([{"params": lora_params, "lr": cfg["lr_lora"]},
                         {"params": head.parameters(), "lr": cfg["lr_head"]}], weight_decay=0.01)
steps = cfg["epochs"] * math.ceil(len(train_items) / cfg["bs"])
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, 0.06 * steps)) * max(0.0, (steps - s) / steps))
scaler = torch.amp.GradScaler("cuda", enabled=AMP)
history = [{"epoch": 0, "val_acc": acc_on(val_items)}]
print(f"epoch 0  val acc {history[0]['val_acc']:.3f}")
best = (history[0]["val_acc"], None)
t0 = time.time()
for ep in range(1, cfg["epochs"] + 1):
    encoder.train(); head.train()
    random.shuffle(train_items)
    tot = 0.0
    for i in range(0, len(train_items), cfg["bs"]):
        batch = train_items[i:i + cfg["bs"]]
        logits, mmask = forward(batch)
        lab = torch.tensor([it["label"] for it in batch], device=DEVICE)
        loss = nn.functional.cross_entropy(logits, lab)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(lora_params + list(head.parameters()), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(batch)
    va = acc_on(val_items)
    history.append({"epoch": ep, "train_loss": tot / len(train_items), "val_acc": va})
    print(f"epoch {ep}  train loss {tot / len(train_items):.4f}  val acc {va:.3f}  ({time.time() - t0:.0f}s)")
    if va >= best[0]:
        best = (va, {"lora": {n: p.detach().cpu().clone() for n, p in encoder.named_parameters() if p.requires_grad},
                     "head": {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}, "epoch": ep})
if best[1] is not None:   # restore the best epoch on the held-out train split (never the eval slice)
    sd = dict(encoder.named_parameters())
    with torch.no_grad():
        for n, v in best[1]["lora"].items():
            sd[n].copy_(v.to(DEVICE))
    head.load_state_dict({k: v.to(DEVICE) for k, v in best[1]["head"].items()})
    print("kept epoch", best[1]["epoch"])
else:
    print("no epoch beat the base model on val; kept the base weights")

epoch 0  val acc 0.747


epoch 1  train loss 0.4967  val acc 0.793  (943s)


epoch 2  train loss 0.4901  val acc 0.793  (1868s)


epoch 3  train loss 0.4781  val acc 0.800  (2788s)
kept epoch 3


## 6. Results on the eval slice

In [8]:
p1 = p_yes(eval_items)
tuned = metrics(p1, eval_y)
ok0, ok1 = (p0 >= .5) == eval_y, (p1 >= .5) == eval_y
fixed, broke = int((~ok0 & ok1).sum()), int((ok0 & ~ok1).sum())
from scipy.stats import binomtest
pval = binomtest(fixed, fixed + broke, 0.5).pvalue if fixed + broke else 1.0
table = pd.DataFrame({
    "Laya zero-shot (this run)": [zero["accuracy"], zero["f1"], zero["precision"], zero["recall"]],
    f"Laya + LoRA top-{cfg['top_layers']} (this run)": [tuned["accuracy"], tuned["f1"], tuned["precision"], tuned["recall"]],
}, index=["accuracy", "F1", "precision", "recall"])
display(table.style.format("{:.3f}"))
print(f"fixed {fixed}, broke {broke}, McNemar p = {pval:.4g}")
print("For reference (published write-up, same slice recipe, different run): Jev 79.3%, "
      "GLiNER2.5-Decide 69.7% zero-shot and 80.3% with LoRA.")
print("confusion before", zero["confusion"], "| after", tuned["confusion"])

,Laya zero-shot (this run),Laya + LoRA top-4 (this run)
accuracy,0.772,0.780
F1,0.767,0.791
precision,0.783,0.752
recall,0.752,0.835


fixed 62, broke 50, McNemar p = 0.2986
For reference (published write-up, same slice recipe, different run): Jev 79.3%, GLiNER2.5-Decide 69.7% zero-shot and 80.3% with LoRA.
confusion before {'tp': 564, 'fp': 156, 'fn': 186, 'tn': 594} | after {'tp': 626, 'fp': 206, 'fn': 124, 'tn': 544}


## 7. Save the adapter and head
Only the LoRA weights (a few MB) and the head are saved; the frozen encoder comes from the base checkpoint.

In [9]:
from safetensors.torch import save_file
out_dir = "laya_lora_qqp"
os.makedirs(out_dir, exist_ok=True)
save_file({n: p.detach().cpu().contiguous() for n, p in encoder.named_parameters() if p.requires_grad}, f"{out_dir}/lora.safetensors")
save_file({k: v.detach().cpu().contiguous() for k, v in head.state_dict().items()}, f"{out_dir}/head.safetensors")
json.dump({"base_model": "convaiinnovations/laya-typed-decisions", "lora_layers": lora_layers,
           "targets": ["attn.Wqkv", "attn.Wo", "mlp.Wi", "mlp.Wo"], **cfg, "history": history,
           "eval": {"zero_shot": zero, "lora": tuned, "fixed": fixed, "broke": broke, "mcnemar_p": pval}},
          open(f"{out_dir}/config.json", "w"), indent=1, default=float)
print(os.listdir(out_dir), sum(os.path.getsize(f"{out_dir}/{f}") for f in os.listdir(out_dir)) / 1e6, "MB")
try:
    from google.colab import files
    import shutil; shutil.make_archive(out_dir, "zip", out_dir); files.download(out_dir + ".zip")
except ImportError:
    pass

['head.safetensors', 'config.json', 'lora.safetensors'] 110.168529 MB


## Notes
* **How this fits the MoE.** Only the top layers change, so a request can run the shared bottom 24 layers once and branch into each expert's top-layer LoRA + head. That keeps one shared trunk while letting experts adapt the encoder.
* **Honest comparison.** The eval slice follows the published recipe, but prompts, rows and hardware may differ from that write-up; compare within this notebook first (zero-shot vs LoRA on identical rows).
* The best epoch is chosen on a held-out part of the *training* pairs, never on the eval slice.
* Modified derivative of `laya-typed-decisions` (Apache-2.0, ConvAI Innovations); unofficial, not affiliated. QQP is © Quora, used here for evaluation under its original terms.